In [1]:
import os
import pandas as pd
import numpy as np
from sklearn.feature_selection import (
    SelectKBest, chi2, f_classif, mutual_info_classif, VarianceThreshold
)
from sklearn.decomposition import PCA
from sklearn.preprocessing import MinMaxScaler, StandardScaler
import warnings

warnings.filterwarnings("ignore")

INPUT_PATH = r"C:\Users\Gonzalo\Documents\GitHub\TesisAustral\Archivos_tesis\intermediate\featuring"
OUTPUT_PATH = r"C:\Users\Gonzalo\Documents\GitHub\TesisAustral\Archivos_tesis\intermediate\selected"
os.makedirs(OUTPUT_PATH, exist_ok=True)

def seleccionar_variables(X_train, y_train, metodo):
    if metodo == "CHI2":
        X_train_nonneg = MinMaxScaler().fit_transform(X_train)
        scores = chi2(X_train_nonneg, y_train)[0]
    elif metodo == "ANOVA":
        scores = f_classif(X_train, y_train)[0]
    elif metodo == "MI":
        scores = mutual_info_classif(X_train, y_train)
    elif metodo == "VAR":
        selector = VarianceThreshold()
        selector.fit(X_train)
        mask = selector.get_support()
        cols = X_train.columns[mask]
        return cols, pd.Series(selector.variances_, index=X_train.columns)
    else:
        raise ValueError("Método de selección no reconocido")

    # Selección dinámica por umbral
    threshold = 0.05 * np.nanmax(scores)
    selected_mask = scores >= threshold
    selected_features = X_train.columns[selected_mask]
    scores_series = pd.Series(scores, index=X_train.columns)
    return selected_features, scores_series

def aplicar_pca(X_train, X_test, n_components, modo="fijo"):
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)

    if modo == "fijo":
        pca = PCA(n_components=n_components)
        X_train_pca = pca.fit_transform(X_train_scaled)
        X_test_pca = pca.transform(X_test_scaled)
        scores = pd.Series(pca.explained_variance_ratio_, index=[f"PC{i+1}" for i in range(n_components)])
    elif modo == "optimo":
        pca_full = PCA().fit(X_train_scaled)
        cumsum = np.cumsum(pca_full.explained_variance_ratio_)
        n_opt = np.argmax(cumsum >= 0.95) + 1
        pca = PCA(n_components=n_opt)
        X_train_pca = pca.fit_transform(X_train_scaled)
        X_test_pca = pca.transform(X_test_scaled)
        scores = pd.Series(pca.explained_variance_ratio_, index=[f"PC{i+1}" for i in range(n_opt)])
    else:
        raise ValueError("Modo PCA no reconocido")

    return pd.DataFrame(X_train_pca), pd.DataFrame(X_test_pca), scores

# Procesar todas las combinaciones
metodos = ["CHI2", "ANOVA", "MI", "VAR", "ALL"]
resumen_comparativo = []

for carpeta in os.listdir(INPUT_PATH):
    carpeta_path = os.path.join(INPUT_PATH, carpeta)
    if not os.path.isdir(carpeta_path):
        continue

    for tipo in ["ORIGINAL", "FE"]:
        nombre_dataset = f"{carpeta}_{tipo}"
        try:
            X_train = pd.read_parquet(f"{carpeta_path}/X_train_{carpeta}_{tipo}.parquet")
            X_test = pd.read_parquet(f"{carpeta_path}/X_test_{carpeta}_{tipo}.parquet")
            y_train = pd.read_parquet(f"{carpeta_path}/y_train_{carpeta}_{tipo}.parquet")
            y_test = pd.read_parquet(f"{carpeta_path}/y_test_{carpeta}_{tipo}.parquet")
        except Exception as e:
            print(f"❌ Error cargando {nombre_dataset}: {e}")
            continue

        print(f"\n🔍 Procesando {nombre_dataset}...")

        # Métodos clásicos de selección
        for metodo in metodos:
            if metodo == "CHI2" and (X_train < 0).any().any():
                print(f"⚠️ Saltando CHI2 por contener negativos en {nombre_dataset}")
                continue

            if metodo == "ALL":
                selected_columns = X_train.columns
                scores = pd.Series([1.0] * len(selected_columns), index=selected_columns)
            else:
                try:
                    selected_columns, scores = seleccionar_variables(X_train, y_train.values.ravel(), metodo)
                except Exception as e:
                    print(f"❌ Error aplicando {metodo} en {nombre_dataset}: {e}")
                    continue

            # Guardar métricas
            scores.to_csv(os.path.join(OUTPUT_PATH, f"metricas_{nombre_dataset}_{metodo}.csv"), header=["score"])

            # Guardar datasets
            X_train_sel = X_train[selected_columns]
            X_test_sel = X_test[selected_columns.intersection(X_test.columns)]
            X_train_sel.to_parquet(os.path.join(OUTPUT_PATH, f"X_train_{nombre_dataset}_{metodo}.parquet"))
            X_test_sel.to_parquet(os.path.join(OUTPUT_PATH, f"X_test_{nombre_dataset}_{metodo}.parquet"))
            y_train.to_parquet(os.path.join(OUTPUT_PATH, f"y_train_{nombre_dataset}_{metodo}.parquet"))
            y_test.to_parquet(os.path.join(OUTPUT_PATH, f"y_test_{nombre_dataset}_{metodo}.parquet"))

            print(f"✔ {nombre_dataset} - {metodo}: {len(selected_columns)} variables seleccionadas")

            resumen_comparativo.append({
                "dataset": nombre_dataset,
                "metodo": metodo,
                "columnas_seleccionadas": len(selected_columns)
            })

        # PCA30 y PCAopt
        for pca_tipo, modo in [("PCA30", "fijo"), ("PCAopt", "optimo")]:
            try:
                X_train_pca, X_test_pca, pca_scores = aplicar_pca(X_train, X_test, n_components=30, modo=modo)

                # Guardar datasets
                X_train_pca.to_parquet(os.path.join(OUTPUT_PATH, f"X_train_{nombre_dataset}_{pca_tipo}.parquet"))
                X_test_pca.to_parquet(os.path.join(OUTPUT_PATH, f"X_test_{nombre_dataset}_{pca_tipo}.parquet"))
                y_train.to_parquet(os.path.join(OUTPUT_PATH, f"y_train_{nombre_dataset}_{pca_tipo}.parquet"))
                y_test.to_parquet(os.path.join(OUTPUT_PATH, f"y_test_{nombre_dataset}_{pca_tipo}.parquet"))

                # Guardar métricas
                pca_scores.to_csv(os.path.join(OUTPUT_PATH, f"metricas_{nombre_dataset}_{pca_tipo}.csv"), header=["explained_variance_ratio"])

                print(f"✔ {nombre_dataset} - {pca_tipo}: {X_train_pca.shape[1]} componentes")
                resumen_comparativo.append({
                    "dataset": nombre_dataset,
                    "metodo": pca_tipo,
                    "columnas_seleccionadas": X_train_pca.shape[1]
                })
            except Exception as e:
                print(f"❌ Error en PCA {pca_tipo} para {nombre_dataset}: {e}")

# Guardar resumen general
df_resumen = pd.DataFrame(resumen_comparativo)
df_resumen.to_csv(os.path.join(OUTPUT_PATH, "resumen_cantidad_variables.csv"), index=False)
print("\n📊 Comparativo guardado en resumen_cantidad_variables.csv")



🔍 Procesando MaxAbs_ORIGINAL...
⚠️ Saltando CHI2 por contener negativos en MaxAbs_ORIGINAL
✔ MaxAbs_ORIGINAL - ANOVA: 118 variables seleccionadas
✔ MaxAbs_ORIGINAL - MI: 126 variables seleccionadas
✔ MaxAbs_ORIGINAL - VAR: 539 variables seleccionadas
✔ MaxAbs_ORIGINAL - ALL: 539 variables seleccionadas
✔ MaxAbs_ORIGINAL - PCA30: 30 componentes
✔ MaxAbs_ORIGINAL - PCAopt: 472 componentes

🔍 Procesando MaxAbs_FE...
⚠️ Saltando CHI2 por contener negativos en MaxAbs_FE
✔ MaxAbs_FE - ANOVA: 287 variables seleccionadas
✔ MaxAbs_FE - MI: 350 variables seleccionadas
✔ MaxAbs_FE - VAR: 980 variables seleccionadas
✔ MaxAbs_FE - ALL: 980 variables seleccionadas
✔ MaxAbs_FE - PCA30: 30 componentes
✔ MaxAbs_FE - PCAopt: 476 componentes
❌ Error cargando metricas_generacion_ORIGINAL: [Errno 2] No such file or directory: 'C:\\Users\\Gonzalo\\Documents\\GitHub\\TesisAustral\\Archivos_tesis\\intermediate\\featuring\\metricas_generacion/X_train_metricas_generacion_ORIGINAL.parquet'
❌ Error cargando metr